# AI Quality Control — Exploration Notebook
Data exploration, model evaluation, and result visualisation.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import yaml, json, cv2

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

print('Config loaded. Classes:', config['classes'])

## 1. Dataset Statistics

In [ ]:
from src.preprocessing.image_processor import ProductDataset, get_train_transforms

data_dir = config['paths']['data_dir']
dataset = ProductDataset(data_dir)

labels = [s[1] for s in dataset.samples]
unique, counts = np.unique(labels, return_counts=True)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['PASS', 'DEFECT'], counts, color=['#3fb950', '#f85149'])
ax.bar_label(bars, padding=3)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Total samples: {len(dataset)}  |  PASS: {counts[0]}  |  DEFECT: {counts[1]}')

## 2. Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
class_names = ['PASS', 'DEFECT']

for row, cls in enumerate(class_names):
    samples = [s for s in dataset.samples if s[1] == row][:4]
    for col, (path, label) in enumerate(samples):
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls, color='green' if cls=='PASS' else 'red')
        axes[row][col].axis('off')

plt.suptitle('Sample Product Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Training History

In [ ]:
history_path = '../models/training_history.png'
if Path(history_path).exists():
    from PIL import Image
    img = Image.open(history_path)
    plt.figure(figsize=(14, 4))
    plt.imshow(img); plt.axis('off')
    plt.title('Training History')
    plt.show()
else:
    print('No training history found — run train.py first.')

## 4. Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
from src.preprocessing.image_processor import get_val_transforms
import torch.nn.functional as F

model_path = '../models/best_model.pth'
if not Path(model_path).exists():
    print('No trained model found — run train.py first.')
else:
    from src.model.cnn_model import QualityControlCNN, load_checkpoint
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = QualityControlCNN(num_classes=2, pretrained=False).to(device)
    load_checkpoint(model, model_path, device)
    model.eval()

    val_ds = ProductDataset(data_dir, transform=get_val_transforms(224))
    loader = DataLoader(val_ds, batch_size=16)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits, _ = model(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['PASS','DEFECT'], yticklabels=['PASS','DEFECT'])
    plt.title('Confusion Matrix')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()

    print(classification_report(all_labels, all_preds, target_names=['PASS','DEFECT']))

## 5. Grad-CAM Visualisation

In [ ]:
# Visualise what regions the model focuses on for defect detection
from src.quality_control.inspector import QualityInspector

inspector = QualityInspector('../models/best_model.pth', config)

test_image = 'path/to/defect/image.jpg'  # <-- change this
result = inspector.inspect(test_image)

print(f"Verdict: {result['verdict']}  ({result['confidence']:.1%} confidence)")
if result['heatmap_path']:
    heatmap = cv2.imread(result['heatmap_path'])
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    orig = cv2.imread(test_image); orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    axes[0].imshow(orig); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(heatmap); axes[1].set_title('Grad-CAM Heatmap'); axes[1].axis('off')
    plt.tight_layout(); plt.show()